In [5]:
!pip install -q google-generativeai arxiv fpdf

In [19]:
import os
import google.generativeai as genai

genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model = genai.GenerativeModel("gemini-2.5-flash")

In [ ]:
import arxiv
from fpdf import FPDF

In [17]:
def topic_refiner(topic):
    prompt = f"""
Rewrite the following as ONLY a short arXiv search query.

Rules:
- Return ONLY the query.
- No explanations.
- Maximum 8 words.

Topic:
{topic}
"""

    response = model.generate_content(prompt)
    return response.text.strip()

In [15]:
def search_papers(topic):

    client = arxiv.Client()

    search = arxiv.Search(
        query=topic,
        max_results=5,
        sort_by=arxiv.SortCriterion.Relevance
    )

    papers = []

    for paper in client.results(search):

        papers.append({
            "title": paper.title,
            "summary": paper.summary,
            "authors": [author.name for author in paper.authors],
            "url": paper.entry_id
        })

    return papers

In [9]:
def summarize_paper(paper):

    prompt=f"""
Summarize this research paper.

Title:
{paper['title']}

Abstract:
{paper['summary']}
"""

    response=model.generate_content(prompt)

    return response.text

In [10]:
def create_report(papers):

    report=""

    for paper in papers:

        summary=summarize_paper(paper)

        report+=f"""

Title:
{paper['title']}

Authors:
{', '.join(paper['authors'])}

Summary:
{summary}

Paper:
{paper['url']}

{'='*80}

"""

    return report

In [11]:
def gap_analysis(report):

    prompt=f"""
Find:

1. Research gaps

2. Limitations

3. Future work

Based on this report:

{report}
"""

    response=model.generate_content(prompt)

    return response.text

In [12]:
choice=input("Generate Final Report? (yes/no): ")

if choice.lower()!="yes":
    raise Exception("Stopped")

Generate Final Report? (yes/no): yes


In [13]:
def save_pdf(text):

    pdf=FPDF()

    pdf.add_page()

    pdf.set_font("Arial",size=12)

    pdf.multi_cell(0,8,text)

    pdf.output("Research_Report.pdf")

In [18]:
topic=input("Enter Research Topic: ")

refined=topic_refiner(topic)

print("Refined Topic:")
print(refined)

papers=search_papers(topic)

report=create_report(papers)

gap=gap_analysis(report)

final_report=report+"\n\nResearch Gap Analysis\n\n"+gap

print(final_report)

save_pdf(final_report)

print("Project Completed Successfully!")

Enter Research Topic: Plant Disease Detection using Deep Learning
Refined Topic:
Deep Learning Plant Disease Detection


Title:
Oriented object detection in optical remote sensing images using deep learning: a survey

Authors:
Kun Wang, Zi Wang, Zhang Li, Ang Su, Xichao Teng, Erting Pan, Minhao Liu, Qifeng Yu

Summary:
This survey paper provides a comprehensive review of recent deep learning advancements in **oriented object detection** specifically for **optical remote sensing images**.

The paper outlines the evolution from horizontal to oriented object detection, detailing key challenges such as feature misalignment, spatial misalignment, oriented bounding box (OBB) regression problems, and general issues in remote sensing. It then categorizes and discusses existing deep learning methods based on detection frameworks, OBB regression techniques, and feature representation approaches, explaining how they tackle these challenges.

Additionally, the survey covers publicly available data